# Headway 분석·정밀화 (Jupyter)

`silver.subway_arrival_events` 에서 역·방향·시간대별 배차 간격(headway)을 구하고,
**표본 기준·컷오프를 바꿔가며** 불안정 역 순위를 인터랙티브하게 본다.
(배치 마트는 `labs/13-spark-headway` — 여기선 탐색·정밀화용)

In [1]:
from pyspark.sql import SparkSession

PKGS = ",".join([
    "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1",
    "org.apache.iceberg:iceberg-aws-bundle:1.6.1",
    "org.apache.paimon:paimon-spark-3.5_2.12:1.4.1",
    "org.apache.paimon:paimon-s3:1.4.1",
])
NETTY = "-Dorg.apache.iceberg.shaded.io.netty.noUnsafe=true -Dio.netty.noUnsafe=true"

spark = (
    SparkSession.builder.appName("headway-analysis")
    .config("spark.jars.packages", PKGS)
    .config("spark.driver.extraJavaOptions", NETTY)
    .config("spark.sql.iceberg.vectorization.enabled", "false")
    .config("spark.sql.catalog.paimon", "org.apache.paimon.spark.SparkCatalog")
    .config("spark.sql.catalog.paimon.warehouse", "s3://paimon/warehouse")
    .config("spark.sql.catalog.paimon.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.paimon.s3.access-key", "minioadmin")
    .config("spark.sql.catalog.paimon.s3.secret-key", "minioadmin")
    .config("spark.sql.catalog.paimon.s3.path.style.access", "true")
    .config(
        "spark.sql.extensions",
        "org.apache.paimon.spark.extensions.PaimonSparkSessionExtensions",
    )
    .getOrCreate()
)
spark.table("paimon.silver.subway_arrival_events").createOrReplaceTempView("arr_src")
print("도착 이벤트:", spark.table("arr_src").count())

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/spark/.ivy2/cache
The jars for the packages stored in: /home/spark/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
org.apache.paimon#paimon-spark-3.5_2.12 added as a dependency
org.apache.paimon#paimon-s3 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-93ca9984-6608-46cb-aa5a-fd714fcb1f7f;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.1 in central
	found org.apache.iceberg#iceberg-aws-bundle;1.6.1 in central
	found org.apache.paimon#paimon-spark-3.5_2.12;1.4.1 in central
	found org.apache.paimon#paimon-format;1.4.1 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;1.7.32 in central
	found com.google.code.findbugs#jsr305;1.3.9 in central
	found org.apache.spark#spark-hive_2.12;3.5.8 in central
	found org.apache.hive#hive-common;2.3.9 in centra

도착 이벤트: 14960


## 파라미터 — 여기 숫자만 바꾸면 즉시 재계산
- `MIN_SAMPLE` : 그룹 최소 표본 수 (1호선 저표본 노이즈 컷)
- `MAX_HEADWAY_SEC` : 이 초과 간격은 윈도우 경계/분기 빈틈으로 제외

In [6]:
MIN_SAMPLE = 15
MAX_HEADWAY_SEC = 600   # 10분

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW headways AS
SELECT * FROM (
  SELECT line, statn_id, statn_nm,
    CASE updn_line WHEN '0' THEN '상행' WHEN '1' THEN '하행' ELSE '미상' END AS direction,
    CASE WHEN hour(arrival_ts) BETWEEN 5 AND 6  THEN '새벽'
         WHEN hour(arrival_ts) BETWEEN 7 AND 9  THEN '출근'
         WHEN hour(arrival_ts) BETWEEN 10 AND 16 THEN '점심'
         WHEN hour(arrival_ts) BETWEEN 17 AND 19 THEN '퇴근'
         ELSE '밤' END AS time_band,
    unix_timestamp(arrival_ts) - unix_timestamp(
      LAG(arrival_ts) OVER (PARTITION BY line, statn_id, updn_line ORDER BY arrival_ts)
    ) AS headway_sec
  FROM arr_src WHERE updn_line IN ('0','1')
)
WHERE headway_sec > 0 AND headway_sec <= {MAX_HEADWAY_SEC}
""")

gold = spark.sql(f"""
SELECT line, statn_nm, direction, time_band,
  COUNT(*) AS n,
  ROUND(percentile_approx(headway_sec,0.5),0) AS p50_sec,
  ROUND(percentile_approx(headway_sec,0.9),0) AS p90_sec,
  ROUND(STDDEV(headway_sec)/NULLIF(AVG(headway_sec),0),3) AS cv
FROM headways
GROUP BY line, statn_id, statn_nm, direction, time_band
HAVING COUNT(*) >= {MIN_SAMPLE}
""")
gold.createOrReplaceTempView("gold")
print("그룹 수:", gold.count())

26/06/24 11:29:31 WARN HadoopUtils: Could not find Hadoop configuration via any of the supported methods
26/06/24 11:29:31 WARN HadoopUtils: Could not find Hadoop configuration via any of the supported methods
26/06/24 11:29:31 WARN HadoopUtils: Could not find Hadoop configuration via any of the supported methods
26/06/24 11:29:31 WARN HadoopUtils: Could not find Hadoop configuration via any of the supported methods


그룹 수: 425


## 출퇴근 배차 불안정 역 Top (CV 기준)

In [7]:
(spark.sql("""
  SELECT line, statn_nm, direction, time_band, n, p50_sec, p90_sec, cv
  FROM gold WHERE time_band IN ('출근','퇴근')
  ORDER BY cv DESC LIMIT 20
""").toPandas())

26/06/24 11:29:54 WARN HadoopUtils: Could not find Hadoop configuration via any of the supported methods
26/06/24 11:29:54 WARN HadoopUtils: Could not find Hadoop configuration via any of the supported methods
26/06/24 11:29:54 WARN HadoopUtils: Could not find Hadoop configuration via any of the supported methods
26/06/24 11:29:54 WARN HadoopUtils: Could not find Hadoop configuration via any of the supported methods


,line,statn_nm,direction,time_band,n,p50_sec,p90_sec,cv
0,9호선,양천향교,상행,퇴근,15,92,525,0.957
1,9호선,가양,하행,출근,21,88,354,0.799
2,9호선,여의도,하행,퇴근,31,101,377,0.767
3,9호선,여의도,하행,출근,21,82,370,0.752
4,9호선,당산,하행,출근,21,99,364,0.749
5,1호선,영등포,하행,출근,29,153,313,0.748
6,9호선,고속터미널,하행,출근,21,116,358,0.739
7,9호선,고속터미널,하행,퇴근,29,206,388,0.727
8,9호선,고속터미널,상행,출근,16,154,481,0.726
9,1호선,구로,상행,출근,26,122,270,0.722


## 노선별·시간대별 평균 CV (러시 vs 점심 비교)

In [8]:
(spark.sql("""
  SELECT line, time_band,
    COUNT(*) AS groups,
    ROUND(AVG(cv),3) AS avg_cv,
    ROUND(AVG(p50_sec),0) AS avg_p50_sec
  FROM gold GROUP BY line, time_band
  ORDER BY line, time_band
""").toPandas())

26/06/24 11:29:59 WARN HadoopUtils: Could not find Hadoop configuration via any of the supported methods
26/06/24 11:29:59 WARN HadoopUtils: Could not find Hadoop configuration via any of the supported methods
26/06/24 11:29:59 WARN HadoopUtils: Could not find Hadoop configuration via any of the supported methods
26/06/24 11:29:59 WARN HadoopUtils: Could not find Hadoop configuration via any of the supported methods


,line,time_band,groups,avg_cv,avg_p50_sec
0,1호선,점심,5,0.537,201.0
1,1호선,출근,53,0.495,197.0
2,1호선,퇴근,27,0.562,179.0
3,2호선,출근,87,0.251,183.0
4,2호선,퇴근,65,0.312,191.0
5,9호선,점심,57,0.424,255.0
6,9호선,출근,58,0.460,176.0
7,9호선,퇴근,73,0.470,218.0


## 메모
- `MIN_SAMPLE`/`MAX_HEADWAY_SEC` 를 바꿔 셀 재실행 → 즉시 결과 갱신.
- **급행/완행 분리**(9호선 CV 해석)를 하려면 L1(`subway_arrival_events`)에 `direct_at` 컬럼을 추가해야 한다 → labs/12 의 도착 추출에 `direct_at` 보존 후 재적재.
- 확정된 정밀화 파라미터는 배치 마트(`labs/13-spark-headway`)에 반영하면 재현 가능한 gold 가 된다.